# POSEIDON: Direct Sentinel-1 SAR Fallback & Mitigation Pipeline (Juli - Agustus 2026)

Pipeline mitigasi deteksi kapal mandiri via Google Earth Engine (GEE) dan penyaringan data noise menggunakan Isolation Forest terlatih dari data historis Global Fishing Watch (GFW).

## Pengecekan Data GFW

In [ ]:
!pip install gfw-api-python-client

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.6/135.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 44.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pan

In [ ]:
import gfwapiclient as gfw
import pandas as pd
from google.colab import userdata

API = userdata.get('GFW_API')
client = gfw.Client(access_token=API)

WPP711_BBOX = {
    "type": "Polygon",
    "coordinates": [[
        [102.90, -4.16],
        [111.05, -4.16],
        [111.05, 7.74],
        [102.90, 7.74],
        [102.90, -4.16]
    ]]
}

WINDOWS = [
    ("Kontrol Jan 2025", "2025-01-01", "2025-01-05"),
    ("Kontrol Jun 2026 (data terakhir portal)", "2026-06-25", "2026-06-30"),
    ("Uji Jul 2026", "2026-07-01", "2026-07-10"),
    ("Uji Agu 2026 (5-20)", "2026-08-05", "2026-08-20"),
]

async def cek_semua():
    hasil_ringkas = []
    for label, start, end in WINDOWS:
        try:
            result = await client.fourwings.create_sar_presence_report(
                start_date=start,
                end_date=end,
                spatial_resolution="HIGH",
                geojson=WPP711_BBOX,
            )
            items = result.data()
            n = len(items)
            print(f"[{label}] {start} → {end}: {n} baris deteksi")
            hasil_ringkas.append({"window": label, "start": start, "end": end, "n_deteksi": n})

            if n > 0:
                df = pd.DataFrame([item.model_dump() for item in items])
                globals()[f"df_{label.split()[0].lower()}"] = df

        except Exception as e:
            print(f"[{label}] {start} → {end}: ERROR -> {e}")
            hasil_ringkas.append({"window": label, "start": start, "end": end, "n_deteksi": "ERROR", "error": str(e)})

    return pd.DataFrame(hasil_ringkas)

ringkasan = await cek_semua()
print("\nRINGKASAN")
print(ringkasan)

[Kontrol Jan 2025] 2025-01-01 → 2025-01-05: 806 baris deteksi
[Kontrol Jun 2026 (data terakhir portal)] 2026-06-25 → 2026-06-30: 1064 baris deteksi
[Uji Jul 2026] 2026-07-01 → 2026-07-10: 0 baris deteksi
[Uji Agu 2026 (5-20)] 2026-08-05 → 2026-08-20: 0 baris deteksi

RINGKASAN
                                    window       start         end  n_deteksi
0                         Kontrol Jan 2025  2025-01-01  2025-01-05        806
1  Kontrol Jun 2026 (data terakhir portal)  2026-06-25  2026-06-30       1064
2                             Uji Jul 2026  2026-07-01  2026-07-10          0
3                      Uji Agu 2026 (5-20)  2026-08-05  2026-08-20          0


In [ ]:
import asyncio

async def cari_titik_potong():
    tanggal_cek = [
        ("2026-06-28", "2026-06-30"),
        ("2026-07-01", "2026-07-01"),
        ("2026-07-02", "2026-07-02"),
        ("2026-07-03", "2026-07-03"),
        ("2026-07-04", "2026-07-05"),
    ]
    for start, end in tanggal_cek:
        try:
            result = await client.fourwings.create_sar_presence_report(
                start_date=start, end_date=end,
                spatial_resolution="HIGH", geojson=WPP711_BBOX,
            )
            print(f"{start} → {end}: {len(result.data())} baris")
        except Exception as e:
            print(f"{start} → {end}: ERROR -> {e}")

await cari_titik_potong()

2026-06-28 → 2026-06-30: 716 baris
2026-07-01 → 2026-07-01: 0 baris
2026-07-02 → 2026-07-02: 0 baris
2026-07-03 → 2026-07-03: 0 baris
2026-07-04 → 2026-07-05: 0 baris


## Pengecekan Data Google Earth Engine (GEE)

### Inisialisasi GEE & Batas Koordinat WPPNRI 711

In [ ]:
import ee
import pandas as pd
import time

ee.Authenticate()
ee.Initialize(project='darvin-natuna-2025')

BBOX_LON_MIN, BBOX_LON_MAX = 102.90, 111.05
BBOX_LAT_MIN, BBOX_LAT_MAX = -4.16, 7.74

wpp711_geom = ee.Geometry.Rectangle([BBOX_LON_MIN, BBOX_LAT_MIN, BBOX_LON_MAX, BBOX_LAT_MAX])

print(f"GEE terinisialisasi. Bounding box WPP 711: Lat [{BBOX_LAT_MIN}, {BBOX_LAT_MAX}], Lon [{BBOX_LON_MIN}, {BBOX_LON_MAX}]")

GEE terinisialisasi. Bounding box WPP 711: Lat [-4.16, 7.74], Lon [102.9, 111.05]


### Pemuatan Data Acuan Historis GFW & Pelatihan Detektor Anomali (Isolation Forest)

In [ ]:
from sklearn.ensemble import IsolationForest
import pandas as pd
import time, gdown, os

HISTORICAL_DATA_ID = "19L7mhcy_tjy9XYrLm5Gy-uXTKyfEQVMo"
historical_csv_path = "POSEIDON_complete_with_estimates.csv"

if not os.path.exists(historical_csv_path):
    print("Mengunduh data acuan historis GFW via gdown...")
    gdown.download(id=HISTORICAL_DATA_ID, output=historical_csv_path, quiet=False)

df_historis = pd.read_csv(historical_csv_path, low_memory=False)

FITUR_ANOMALI = ['vv_intensity_db', 'vh_intensity_db', 'vv_vh_ratio', 'background_clutter', 'snr_db']

X_train_anomaly = df_historis[FITUR_ANOMALI].dropna()

print(f"Total data acuan historis: {len(df_historis):,} baris")
print(f"Total data latih dengan 5 fitur lengkap: {len(X_train_anomaly):,} baris")

detektor_kapal = IsolationForest(
    n_estimators=200,
    contamination=0.02,
    random_state=42,
    n_jobs=-1
)
detektor_kapal.fit(X_train_anomaly)

print("Model Isolation Forest berhasil dilatih pada data terkonfirmasi GFW.")

Mengunduh data acuan historis GFW via gdown...


Downloading...
From: https://drive.google.com/uc?id=19L7mhcy_tjy9XYrLm5Gy-uXTKyfEQVMo
To: /content/POSEIDON_complete_with_estimates.csv
100%|██████████| 76.0M/76.0M [00:00<00:00, 96.6MB/s]


Total data acuan historis: 151,851 baris
Total data latih dengan 5 fitur lengkap: 151,851 baris
Model Isolation Forest berhasil dilatih pada data terkonfirmasi GFW.


### Query Citra Sentinel-1 SAR GRD IW & Segmentasi Deteksi Kapal di GEE

In [ ]:
TANGGAL_MULAI = '2026-07-01'
TANGGAL_AKHIR = '2026-08-23'
AMBANG_DB = -3.0

koleksi_s1 = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(wpp711_geom)
    .filterDate(TANGGAL_MULAI, TANGGAL_AKHIR)
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
)

n_scene = koleksi_s1.size().getInfo()
print(f"Total scene Sentinel-1 GRD IW ditemukan (Juli - Agustus 2026): {n_scene}")

daftar_scene = koleksi_s1.toList(n_scene)
kandidat_raw_list = []
scene_gagal = []

for i in range(n_scene):
    try:
        img = ee.Image(daftar_scene.get(i))
        scene_id = img.get('system:index').getInfo()
        tgl_scene = img.date().format('YYYY-MM-dd HH:mm:ss').getInfo()

        vv = img.select('VV')
        kandidat_mask = vv.gt(AMBANG_DB)

        blob = kandidat_mask.selfMask().connectedComponents(
            connectedness=ee.Kernel.plus(1),
            maxSize=32
        )

        ukuran_blob = blob.select('labels').connectedPixelCount(maxSize=32)
        blob_valid = blob.updateMask(ukuran_blob.gte(2).And(ukuran_blob.lte(20)))

        area_irisan = img.geometry().intersection(wpp711_geom, ee.ErrorMargin(1))

        vektor = blob_valid.select('labels').reduceToVectors(
            geometry=area_irisan,
            scale=40,
            geometryType='centroid',
            maxPixels=1e9,
            bestEffort=True
        )

        fitur_list = vektor.limit(3000).getInfo()['features']

        for f in fitur_list:
            lon, lat = f['geometry']['coordinates']
            kandidat_raw_list.append({
                'scene_id': scene_id,
                'timestamp': tgl_scene,
                'lat': lat,
                'lon': lon
            })

        print(f"[{i+1}/{n_scene}] Scene {scene_id} ({tgl_scene}): {len(fitur_list)} kandidat terekstraksi")
        time.sleep(0.2)

    except Exception as e:
        print(f"[{i+1}/{n_scene}] Gagal memproses scene index {i}: {e}")
        scene_gagal.append(i)

df_kandidat_raw = pd.DataFrame(kandidat_raw_list)
print(f"\nTotal kandidat deteksi mentah terkumpul: {len(df_kandidat_raw):,} titik dari {n_scene - len(scene_gagal)} scene")

Total scene Sentinel-1 GRD IW ditemukan (Juli - Agustus 2026): 208
[1/208] Scene S1D_IW_GRDH_1SDV_20260701T223832_20260701T223901_003486_0062A4_36AD (2026-07-01 22:38:32): 2830 kandidat terekstraksi
[2/208] Scene S1D_IW_GRDH_1SDV_20260701T223901_20260701T223926_003486_0062A4_125E (2026-07-01 22:39:01): 76 kandidat terekstraksi
[3/208] Scene S1D_IW_GRDH_1SDV_20260701T223926_20260701T223951_003486_0062A4_526B (2026-07-01 22:39:26): 191 kandidat terekstraksi
[4/208] Scene S1D_IW_GRDH_1SDV_20260701T223951_20260701T224016_003486_0062A4_AF56 (2026-07-01 22:39:51): 3000 kandidat terekstraksi
[5/208] Scene S1D_IW_GRDH_1SDV_20260701T224016_20260701T224041_003486_0062A4_60D9 (2026-07-01 22:40:16): 3000 kandidat terekstraksi
[6/208] Scene S1D_IW_GRDH_1SDV_20260701T224041_20260701T224106_003486_0062A4_106B (2026-07-01 22:40:41): 3000 kandidat terekstraksi
[7/208] Scene S1D_IW_GRDH_1SDV_20260703T222221_20260703T222251_003515_0063A9_48DB (2026-07-03 22:22:21): 814 kandidat terekstraksi
[8/208] Scene

### Ekstraksi Fitur Polarimetri Radar 5 Dimensi per Titik Deteksi

In [ ]:
VESSEL_BUFFER_M = 400
BG_INNER_M = 200
BG_OUTER_M = 800

def get_s1_image_by_id(scene_id):
    try:
        asset_path = f'COPERNICUS/S1_GRD/{scene_id}'
        img = ee.Image(asset_path).select(['VV', 'VH'])
        _ = img.get('system:time_start').getInfo()
        return img
    except Exception:
        return None

def extract_features_scene_points(s1_image, df_scene_pts):
    features_vessel = []
    for r_idx, row in df_scene_pts.iterrows():
        pt = ee.Feature(
            ee.Geometry.Point([row['lon'], row['lat']]).buffer(VESSEL_BUFFER_M),
            {'row_idx': r_idx}
        )
        features_vessel.append(pt)
    fc_vessel = ee.FeatureCollection(features_vessel)

    features_bg = []
    for r_idx, row in df_scene_pts.iterrows():
        pt = ee.Feature(
            ee.Geometry.Point([row['lon'], row['lat']])
            .buffer(BG_OUTER_M)
            .difference(ee.Geometry.Point([row['lon'], row['lat']]).buffer(BG_INNER_M)),
            {'row_idx': r_idx}
        )
        features_bg.append(pt)
    fc_bg = ee.FeatureCollection(features_bg)

    vessel_stats = s1_image.reduceRegions(collection=fc_vessel, reducer=ee.Reducer.mean(), scale=10)
    bg_stats = s1_image.select('VV').reduceRegions(collection=fc_bg, reducer=ee.Reducer.mean(), scale=10)

    v_list = vessel_stats.toList(vessel_stats.size()).getInfo()
    bg_list = bg_stats.toList(bg_stats.size()).getInfo()

    v_map = {f['properties']['row_idx']: f['properties'] for f in v_list}
    bg_map = {f['properties']['row_idx']: f['properties'] for f in bg_list}

    extracted_rows = []
    for r_idx, row in df_scene_pts.iterrows():
        v = v_map.get(r_idx, {})
        bg = bg_map.get(r_idx, {})
        vv_val = v.get('VV')
        vh_val = v.get('VH')
        bg_val = bg.get('mean')
        vv_vh_ratio = (vv_val - vh_val) if (vv_val is not None and vh_val is not None) else np.nan
        snr = (vv_val - bg_val) if (vv_val is not None and bg_val is not None) else np.nan

        extracted_rows.append({
            'row_idx': r_idx,
            'vv_intensity_db': vv_val,
            'vh_intensity_db': vh_val,
            'vv_vh_ratio': vv_vh_ratio,
            'background_clutter': bg_val,
            'snr_db': snr
        })
    return extracted_rows

unique_scenes = df_kandidat_raw['scene_id'].drop_duplicates().tolist()
print(f"Memulai ekstraksi fitur polarimetri untuk {len(unique_scenes)} scene unik...")

all_feature_rows = []
for s_idx, scene_id in enumerate(unique_scenes):
    df_sub = df_kandidat_raw[df_kandidat_raw['scene_id'] == scene_id].reset_index(drop=True)
    img_s1 = get_s1_image_by_id(scene_id)
    if img_s1 is None:
        print(f"[{s_idx+1}/{len(unique_scenes)}] Melewati {scene_id} (gagal memuat)")
        continue

    try:
        t0 = time.time()
        res_features = extract_features_scene_points(img_s1, df_sub)
        for rf in res_features:
            base_row = df_sub.iloc[rf['row_idx']].to_dict()
            base_row.update(rf)
            all_feature_rows.append(base_row)
        print(f"[{s_idx+1}/{len(unique_scenes)}] Scene {scene_id}: {len(res_features)} fitur terekstraksi ({time.time()-t0:.1f} detik)")
    except Exception as e:
        print(f"[{s_idx+1}/{len(unique_scenes)}] Error ekstraksi pada scene {scene_id}: {e}")

df_kandidat_fitur = pd.DataFrame(all_feature_rows)
print(f"\nTotal kandidat berhasil diekstrak fiturnya: {len(df_kandidat_fitur):,} baris")

Memulai ekstraksi fitur polarimetri untuk 199 scene unik...
[1/199] Scene S1D_IW_GRDH_1SDV_20260701T223832_20260701T223901_003486_0062A4_36AD: 2830 fitur terekstraksi (39.9 detik)
[2/199] Scene S1D_IW_GRDH_1SDV_20260701T223901_20260701T223926_003486_0062A4_125E: 76 fitur terekstraksi (2.4 detik)
[3/199] Scene S1D_IW_GRDH_1SDV_20260701T223926_20260701T223951_003486_0062A4_526B: 191 fitur terekstraksi (4.8 detik)
[4/199] Scene S1D_IW_GRDH_1SDV_20260701T223951_20260701T224016_003486_0062A4_AF56: 3000 fitur terekstraksi (48.4 detik)
[5/199] Scene S1D_IW_GRDH_1SDV_20260701T224016_20260701T224041_003486_0062A4_60D9: 3000 fitur terekstraksi (41.2 detik)
[6/199] Scene S1D_IW_GRDH_1SDV_20260701T224041_20260701T224106_003486_0062A4_106B: 3000 fitur terekstraksi (59.4 detik)
[7/199] Scene S1D_IW_GRDH_1SDV_20260703T222221_20260703T222251_003515_0063A9_48DB: 814 fitur terekstraksi (10.0 detik)
[8/199] Scene S1D_IW_GRDH_1SDV_20260703T222251_20260703T222316_003515_0063A9_2E73: 3000 fitur terekstraksi

[99/199] Scene S1D_IW_GRDH_1SDV_20260727T222317_20260727T222342_003865_006F93_7FC2: 3000 fitur terekstraksi (44.1 detik)
[100/199] Scene S1D_IW_GRDH_1SDV_20260727T222342_20260727T222407_003865_006F93_D0C1: 3000 fitur terekstraksi (52.2 detik)
[101/199] Scene S1D_IW_GRDH_1SDV_20260727T222407_20260727T222432_003865_006F93_EF16: 284 fitur terekstraksi (5.3 detik)
[102/199] Scene S1D_IW_GRDH_1SDV_20260729T112303_20260729T112328_003887_007059_4586: 236 fitur terekstraksi (3.2 detik)
[103/199] Scene S1D_IW_GRDH_1SDV_20260729T112328_20260729T112353_003887_007059_F53D: 3000 fitur terekstraksi (23.6 detik)
[104/199] Scene S1D_IW_GRDH_1SDV_20260729T112353_20260729T112418_003887_007059_0DF0: 3000 fitur terekstraksi (44.8 detik)
[105/199] Scene S1D_IW_GRDH_1SDV_20260729T112418_20260729T112443_003887_007059_25C9: 3000 fitur terekstraksi (49.2 detik)
[106/199] Scene S1D_IW_GRDH_1SDV_20260729T112443_20260729T112508_003887_007059_2B10: 3000 fitur terekstraksi (54.2 detik)
[107/199] Scene S1D_IW_GRDH_1

### Penyaringan Noise Clutter Laut dengan Isolation Forest

In [ ]:
df_eval_anomaly = df_kandidat_fitur[FITUR_ANOMALI].dropna()
print(f"Jumlah baris dengan 5 fitur lengkap untuk prediksi: {len(df_eval_anomaly):,}")

prediksi_anomali = detektor_kapal.predict(df_eval_anomaly)
skor_keputusan = detektor_kapal.decision_function(df_eval_anomaly)

df_hasil_filter = df_kandidat_fitur.loc[df_eval_anomaly.index].copy()
df_hasil_filter['is_vessel'] = (prediksi_anomali == 1).astype(int)
df_hasil_filter['anomaly_score'] = skor_keputusan

df_terfilter = df_hasil_filter[df_hasil_filter['is_vessel'] == 1].copy().reset_index(drop=True)

print(f"Total kandidat sebelum filtering : {len(df_hasil_filter):,}")
print(f"Total kandidat lolos filter kapal : {len(df_terfilter):,} ({len(df_terfilter)/len(df_hasil_filter)*100:.1f}%)")
print(f"Total noise/clutter dibuang       : {(df_hasil_filter['is_vessel'] == 0).sum():,}")

Jumlah baris dengan 5 fitur lengkap untuk prediksi: 381,563
Total kandidat sebelum filtering : 381,563
Total kandidat lolos filter kapal : 22,080 (5.8%)
Total noise/clutter dibuang       : 359,483


In [ ]:
from sklearn.model_selection import train_test_split
from scipy.stats import ks_2samp

X_train_gf, X_val_gf = train_test_split(X_train_anomaly, test_size=0.2, random_state=42)

detektor_eval = IsolationForest(n_estimators=200, contamination=0.02, random_state=42, n_jobs=-1)
detektor_eval.fit(X_train_gf)

pred_val = detektor_eval.predict(X_val_gf)
inlier_recall = (pred_val == 1).mean() * 100

print(f"HASIL EVALUASI ISOLATION FOREST")
print(f"1. Inlier Retention Rate (Holdout GFW 20%): {inlier_recall:.2f}% (Target >= 98.0%)")

snr_lolos = df_hasil_filter.loc[df_hasil_filter['is_vessel'] == 1, 'snr_db'].dropna()
snr_ditolak = df_hasil_filter.loc[df_hasil_filter['is_vessel'] == 0, 'snr_db'].dropna()

print(f"2. Rata-rata SNR dB Lolos (Kapal) : {snr_lolos.mean():.2f} dB")
print(f"3. Rata-rata SNR dB Ditolak (Noise): {snr_ditolak.mean():.2f} dB")

ks_stat, ks_p = ks_2samp(snr_lolos, snr_ditolak)
print(f"4. Separabilitas KS-Test (p-value)  : Stat={ks_stat:.4f}, p={ks_p:.4e}")

HASIL EVALUASI ISOLATION FOREST
1. Inlier Retention Rate (Holdout GFW 20%): 97.94% (Target >= 98.0%)
2. Rata-rata SNR dB Lolos (Kapal) : 1.21 dB
3. Rata-rata SNR dB Ditolak (Noise): 0.49 dB
4. Separabilitas KS-Test (p-value)  : Stat=0.3860, p=0.0000e+00


### Standardisasi Skema Dataframe & Ekspor Berkas Fallback 2026

In [ ]:
from google.colab import files
import numpy as np

df_terfilter['timestamp'] = pd.to_datetime(df_terfilter['timestamp'], utc=True)
df_terfilter['matched_category'] = 'unmatched'
df_terfilter['ais_matched'] = 0
df_terfilter['presence_score'] = 1.0
df_terfilter['radar_intensity_diff'] = (df_terfilter['vv_intensity_db'] - df_terfilter['vh_intensity_db']).astype(np.float32)

kolom_ekspor = [
    'scene_id', 'timestamp', 'lat', 'lon',
    'vv_intensity_db', 'vh_intensity_db', 'radar_intensity_diff',
    'background_clutter', 'snr_db', 'vv_vh_ratio',
    'presence_score', 'matched_category', 'ais_matched'
]

kolom_tersedia = [c for c in kolom_ekspor if c in df_terfilter.columns]
df_fallback_final = df_terfilter[kolom_tersedia].copy()

output_filename = "POSEIDON_WPP711_fallback_2026.csv"
df_fallback_final.to_csv(output_filename, index=False)

print(f"Berkas hasil mitigasi berhasil disimpan: {output_filename}")
print(f"Total baris deteksi: {len(df_fallback_final):,} baris")
print(f"Rentang waktu: {df_fallback_final['timestamp'].min()} s.d. {df_fallback_final['timestamp'].max()}")

files.download(output_filename)
print("Mengunduh berkas langsung ke perangkat lokal...")

Berkas hasil mitigasi berhasil disimpan: POSEIDON_WPP711_fallback_2026.csv
Total baris deteksi: 22,080 baris
Rentang waktu: 2026-07-01 22:38:32+00:00 s.d. 2026-08-22 11:26:24+00:00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Mengunduh berkas langsung ke perangkat lokal...


In [ ]:
import pandas as pd
import os

if 'df_fallback_final' in locals():
    df_data = df_fallback_final.copy()
elif os.path.exists("POSEIDON_WPP711_fallback_2026.csv"):
    df_data = pd.read_csv("POSEIDON_WPP711_fallback_2026.csv")
elif os.path.exists("fallback_gee_juli_agustus_2026.csv"):
    df_data = pd.read_csv("fallback_gee_juli_agustus_2026.csv")
else:
    raise FileNotFoundError("Data deteksi mitigasi belum ditemukan di session/direktori.")

df_scenes = df_data[['scene_id', 'timestamp']].drop_duplicates().copy()
df_scenes['date'] = pd.to_datetime(df_scenes['timestamp']).dt.date

scenes_per_day = df_scenes.groupby('date')['scene_id'].nunique()

total_scene = df_scenes['scene_id'].nunique()
total_hari_lintasan = len(scenes_per_day)

tgl_awal = df_scenes['date'].min()
tgl_akhir = df_scenes['date'].max()
total_hari_kalender = (tgl_akhir - tgl_awal).days + 1

rata_per_hari_aktif = scenes_per_day.mean()
rata_per_hari_kalender = total_scene / total_hari_kalender

print("STATISTIK RATA-RATA SCENE PER HARI (DARI DATA MITIGASI)")
print(f"Rentang Tanggal: {tgl_awal} s.d. {tgl_akhir}")
print(f"Total Scene Unik: {total_scene} scene")
print(f"Total Hari Lintasan Satelit: {total_hari_lintasan} hari")
print(f"Total Hari Kalender: {total_hari_kalender} hari")
print(f"Rata-rata Scene / Hari Lintasan: {rata_per_hari_aktif:.2f} scene/hari")
print(f"Rata-rata Scene / Hari Kalender: {rata_per_hari_kalender:.2f} scene/hari")
print(f"Rentang Harian: Min {scenes_per_day.min()} | Max {scenes_per_day.max()} scene")
print(f"Median Scene per Hari: {scenes_per_day.median():.1f} scene")
print("\nRincian Jumlah Scene per Tanggal:")
print(scenes_per_day.to_string())

STATISTIK RATA-RATA SCENE PER HARI (DARI DATA MITIGASI)
Rentang Tanggal: 2026-07-01 s.d. 2026-08-22
Total Scene Unik: 190 scene
Total Hari Lintasan Satelit: 30 hari
Total Hari Kalender: 53 hari
Rata-rata Scene / Hari Lintasan: 6.33 scene/hari
Rata-rata Scene / Hari Kalender: 3.58 scene/hari
Rentang Harian: Min 2 | Max 9 scene
Median Scene per Hari: 6.5 scene

Rincian Jumlah Scene per Tanggal:
date
2026-07-01    6
2026-07-03    5
2026-07-05    8
2026-07-06    7
2026-07-07    2
2026-07-10    7
2026-07-12    9
2026-07-13    6
2026-07-15    5
2026-07-17    9
2026-07-18    6
2026-07-20    6
2026-07-22    7
2026-07-24    9
2026-07-27    5
2026-07-29    8
2026-07-31    2
2026-08-03    7
2026-08-04    4
2026-08-05    9
2026-08-06    6
2026-08-08    5
2026-08-10    8
2026-08-11    7
2026-08-12    2
2026-08-15    7
2026-08-17    9
2026-08-18    6
2026-08-20    5
2026-08-22    8


In [ ]:
import joblib
from google.colab import files

joblib.dump(detektor_kapal, "isolation_forest_model.pkl")
print("Model Isolation Forest berhasil disimpan: isolation_forest_model.pkl")

files.download("isolation_forest_model.pkl")

Model Isolation Forest berhasil disimpan: isolation_forest_model.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>